In [ ]:

"""
ETAPA 1: Acesso à Imagem Sentinel-2 para o sudoeste do Pará
Implementação robusta garantindo que o ROI fique totalmente dentro do estado
"""

import ee
import geemap
import json

ee.Initialize(project="desafio-solved")

municipios = ee.FeatureCollection("FAO/GAUL/2015/level2")

lista_municipios = [
    'Altamira', 'Anapu', 'Brasil Novo', 'Medicilandia',
    'Pacaja', 'Senador Jose Porfirio', 'Uruara', 'Vitoria do Xingu',
    'Itaituba', 'Aveiro', 'Jacareacanga', 'Novo Progresso',
    'Ruropolis', 'Trairao'
]

sudoeste = municipios.filter(ee.Filter.inList("ADM2_NAME", lista_municipios))

roi = sudoeste.geometry()

colecao = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

imagem = colecao.median().clip(roi)

mapa = geemap.Map()
mapa.centerObject(roi, 7)

mapa.addLayer(imagem, {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 3000
}, 'Sudoeste do Pará')

mapa


Área de interesse definida (ROI totalmente dentro do Pará)
445 imagens disponíveis

Dados da imagem selecionada:
ID: 20230330T140711_20230330T141324_T21MVN
Data: {'type': 'Date', 'value': 1680185669497}
Cobertura do ROI: 20.6%
Nuvens: 13.126984%
Cobertura insuficiente. Criando mosaico...
Mosaico criado com múltiplas imagens
Imagem recortada para o ROI do sudoeste do Pará

Etapa 1 concluída Metadata salva em 'etapa1_metadata.json'


Map(center=[-7.21096138882846, -56.962456865658595], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
"""
ETAPA 2: Pré-processamento da Imagem Sentinel-2
VERSÃO CORRIGIDA - Recarrega imagem da coleção
"""

import ee
import geemap
import json

print("🔄 Iniciando Etapa 2 - Pré-processamento (versão corrigida)...")

# Inicializar Earth Engine
ee.Initialize(project="desafio-solved")

# ============================================
# 1️⃣ RECRIAR ROI E BUSCAR IMAGEM NOVAMENTE
# ============================================

# Carregar metadados da etapa 1
with open('etapa1_metadata.json', 'r') as f:
    metadata = json.load(f)

# Recriar ROI
roi_coords = metadata['roi_coordinates']
roi = ee.Geometry.Polygon(roi_coords)
print(f"✅ ROI recriado: {roi_coords}")

# BUSCAR IMAGEM DIRETAMENTE DA COLEÇÃO (mais seguro)
print("   ├─ Buscando imagem na coleção Sentinel-2...")

# Recriar a coleção com os mesmos filtros
collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(roi)
    .filterDate('2023-01-01', '2023-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

# Pegar a primeira imagem (como na etapa 1)
final_image = collection.first()

# Verificar se encontrou
image_id = final_image.id().getInfo()
print(f"✅ Nova imagem selecionada: {image_id}")
print(f"   Data: {final_image.date().getInfo()}")
print(f"   Nuvens: {final_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()}%")

# ============================================
# 2️⃣ MASCARAMENTO SIMPLES DE NUVENS
# ============================================

def simple_cloud_mask(image):
    """
    Máscara básica de nuvens usando QA60
    """
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0)
    return image.updateMask(mask)

print("   ├─ Aplicando máscara de nuvens...")
image_masked = simple_cloud_mask(final_image)

# ============================================
# 3️⃣ ÍNDICES PRINCIPAIS
# ============================================

def add_basic_indices(image):
    """
    Apenas NDVI e Brightness (mais simples e rápido)
    """
    # NDVI
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    
    # Brightness Index simplificado
    bi = image.expression(
        '(B4 + B3 + B2) / 3',
        {'B4': image.select('B4'), 'B3': image.select('B3'), 'B2': image.select('B2')}
    ).rename('BI')
    
    return image.addBands([ndvi, bi])

print("   ├─ Calculando índices...")
image_with_indices = add_basic_indices(image_masked)

# ============================================
# 4️⃣ VISUALIZAÇÃO SIMPLES (SEM NORMALIZAÇÃO PESADA)
# ============================================

print("   ├─ Criando visualização...")
Map2 = geemap.Map()
Map2.centerObject(roi, 10)

# RGB Original
vis_rgb = {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}
Map2.addLayer(final_image.clip(roi), vis_rgb, 'Original RGB (recortada)', True)

# NDVI
vis_ndvi = {'bands': ['NDVI'], 'min': -1, 'max': 1, 'palette': ['blue', 'white', 'green']}
Map2.addLayer(image_with_indices.clip(roi), vis_ndvi, 'NDVI', False)

# Brightness
vis_bi = {'bands': ['BI'], 'min': 0, 'max': 3000, 'palette': ['black', 'gray', 'white']}
Map2.addLayer(image_with_indices.clip(roi), vis_bi, 'Brightness', False)

# Áreas potenciais (NDVI baixo + brilho alto)
pista_potencial = image_with_indices.expression(
    '(1 - NDVI) * (BI / 3000)',
    {'NDVI': image_with_indices.select('NDVI'), 'BI': image_with_indices.select('BI')}
).rename('PISTA_POTENCIAL')

vis_pista = {'min': 0, 'max': 1, 'palette': ['black', 'yellow', 'red']}
Map2.addLayer(pista_potencial.clip(roi), vis_pista, 'Áreas Potenciais', True)

Map2.addLayerControl()
Map2

# ============================================
# 5️⃣ SALVAR IMAGEM PROCESSADA (OPCIONAL - MAIS LEVE)
# ============================================

# Selecionar bandas de interesse
bandas_finais = ['B2', 'B3', 'B4', 'B8', 'NDVI', 'BI']
imagem_final = image_with_indices.select(bandas_finais).clip(roi)

print("\n📊 RESUMO:")
print(f"   ├─ Bandas disponíveis: {imagem_final.bandNames().size().getInfo()}")
print(f"   ├─ Bandas selecionadas: {bandas_finais}")
print(f"   └─ Imagem recortada para o ROI")

# Salvar metadados atualizados
metadata_etapa2 = {
    'etapa': '2_preprocessamento_corrigido',
    'imagem_id': image_id,
    'bandas': bandas_finais,
    'indices': ['NDVI', 'BI']
}

with open('etapa2_metadata.json', 'w') as f:
    json.dump(metadata_etapa2, f, indent=2)

print("\n✅ Etapa 2 concluída!")
print("✅ Metadata salva em 'etapa2_metadata.json'")

# Exportar se desejar
if input("\nDeseja exportar a imagem para o Drive? (s/n): ").lower() == 's':
    task = ee.batch.Export.image.toDrive(
        image=imagem_final,
        description='sentinel2_processada',
        folder='desafio_pistas',
        scale=20,
        region=roi,
        maxPixels=1e10
    )
    task.start()
    print("✅ Exportação iniciada! Verifique no Google Drive")

🔄 Iniciando Etapa 2 - Pré-processamento (versão corrigida)...
✅ ROI recriado: [[[-58.00000000000001, -7.490242262328723], [-57.99325346765701, -7.500056366601279], [-57.99194243639728, -7.505251195484076], [-57.986667327691784, -7.514802623649774], [-57.975889674269425, -7.528099612279084], [-57.97063685092911, -7.537173930094357], [-57.96389024600286, -7.55454212896449], [-57.95155195242764, -7.579829660486547], [-57.94460909046141, -7.601844296006833], [-57.941782021855815, -7.618463357566716], [-57.941750806933804, -7.642323987385667], [-57.939053091169754, -7.652820723190459], [-57.93694390265808, -7.655112694690036], [-57.93388944092346, -7.65645485990555], [-57.92250088582171, -7.655995615458148], [-57.9156963264854, -7.657025611463567], [-57.90889167411568, -7.6607266816486765], [-57.90094558013291, -7.668012890427155], [-57.8984440676392, -7.671575703448869], [-57.89625012383021, -7.67755976713566], [-57.8891423218556, -7.7088225806211135], [-57.886779051493946, -7.727296599192